# Part 1 - Build the Data Subset (50 Most Common PartID's)

**P2 shape classifier**

Builds the classifier's training and evaluation data from the Gdańsk *LEGO bricks for training classification network* dataset (Boiński, Zaraziński & Śledź, 2021; CC BY; DOI 10.34808/rcza-jy08):

1. Downloads the full dataset
2. Extracts the 50 parts listed in `P2/classes_50.csv`, both real photos and renders. List chosen from brickarchitect.org and the availability of training data
3. Counts images per part.
4. Splits the **real photos** 70/15/15 into train/dev/test. Renders are used for training only. (70% for training, 15% for development, and 15% for test).
5. Saves the subset (`lego50.zip`) and the split files to Google Drive under `MyDrive/ece539-p2/`.

## Settings

In [ ]:
from pathlib import Path

SEED = 539                  # fixed so the split is identical on every run
SPLIT = (0.70, 0.15, 0.15)  # train / dev / test share of the REAL photos (70%, 15%, 15%)
MIN_PHOTOS = 30             # parts with fewer real photos than this get flagged

DATASET_URL = ("https://mostwiedzy.pl/en/open-research-data/"
               "lego-bricks-for-training-classification-network,202309140842198941751-0/download")
DATASET_BYTES = 6_469_779_381 #6.46GB
REPO_URL = "https://github.com/divijsinghkohli/cs539-backprop-boys.git"

WORK = Path("/content") #A base folder for temporary work
FULL_ZIP = WORK / "dataset.zip"      # full download on Colab
SUBSET_DIR = WORK / "lego50"         # extracted 50-part subset
REPO_DIR = WORK / "repo" #Copy of the team GitHub Repo
DRIVE_DIR = Path("/content/drive/MyDrive/ece539-p2")
RESULTS_DIR = DRIVE_DIR / "results"

## Mount Drive and load the class list

Read the 50 most common part list

In [ ]:
import subprocess
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
(RESULTS_DIR / "splits").mkdir(parents=True, exist_ok=True) #Create splits/ subdirectory

uploaded = WORK / "classes_50.csv"
if uploaded.exists(): #file name must match
    CLASSES_CSV = uploaded
else:
    #try to get it
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    CLASSES_CSV = REPO_DIR / "P2" / "classes_50.csv"

assert CLASSES_CSV.exists(), (
    "Top 50 Classes Csv not found")

classes = pd.read_csv(CLASSES_CSV, dtype=str)
classes["class_idx"] = classes["class_idx"].astype(int)
assert len(classes) == 50 and classes["dataset_folder"].is_unique

WANTED = set(classes["dataset_folder"]) #dataset_folder gives the location inside the dataset
folder_to_idx = dict(zip(classes["dataset_folder"], classes["class_idx"])) #mapping between folders and indexes
folder_to_part = dict(zip(classes["dataset_folder"], classes["part_id_current"]))
print(f"Loaded {len(classes)} classes from {CLASSES_CSV}")
classes.head() #returns the first 5 rows

## Download the dataset

Downloads the full dataset zip using an allowed script

In [ ]:
import shutil

#Make sure there is enough available storage on the VM
# / 1e9 to convert bytes to GB
free_gb = shutil.disk_usage(WORK).free / 1e9 
print(f"Free disk on this Colab machine: {free_gb:.0f} GB")

if FULL_ZIP.exists() and FULL_ZIP.stat().st_size == DATASET_BYTES:
    print("Already downloaded, skipping.")
else:
    assert free_gb > 10, "Not enough free disk for the 6.5 GB zip plus the subset."
    get_ipython().system(
        f'wget -c -q --show-progress --progress=bar:force:noscroll -O "{FULL_ZIP}" "{DATASET_URL}"')

size = FULL_ZIP.stat().st_size #should be ~6.5GB
assert size == DATASET_BYTES, (
    f"Download incomplete: {size:,} of {DATASET_BYTES:,} bytes. Re-run this cell to resume.")
print(f"OK: {size / 1e9:.2f} GB")

## Look inside the zip

Images are stored under `photos/<part>/` and `renders/<part>/`, possibly below a top-level folder.
This step locates them and checks that all 50 parts are present in both.

In [ ]:
import zipfile
from collections import defaultdict
from pathlib import PurePosixPath

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

zf = zipfile.ZipFile(FULL_ZIP)
infos = [i for i in zf.infolist() if not i.is_dir() and "__MACOSX" not in i.filename]
print(f"{len(infos):,} files in the zip. First few:")
for i in infos[:8]:
    print("   ", i.filename)

def parse(name):
    # 'any/prefix/photos/3001/sub/img.jpg' -> ('photo', '3001', 'sub/img.jpg'); None if not an image
    parts = PurePosixPath(name).parts
    if PurePosixPath(name).suffix.lower() not in IMG_EXT:
        return None
    for k, p in enumerate(parts[:-2]):
        if p.lower() in ("photos", "renders"):
            return p.lower()[:-1], parts[k + 1], "/".join(parts[k + 2:])
    return None

folders = defaultdict(set)   # 'photo'/'render' -> part folders present
for i in infos:
    p = parse(i.filename)
    if p:
        folders[p[0]].add(p[1])

print(f"\nPart folders found: {len(folders['photo'])} under photos/, {len(folders['render'])} under renders/")
for kind in ("photo", "render"):
    missing = sorted(WANTED - folders[kind])
    if missing:
        print(f"MISSING from {kind}s/: {missing}")
        for m in missing:
            stem = m.rstrip("abcdefghijklmnopqrstuvwxyz")
            near = sorted(f for f in folders[kind] if f.startswith(stem))
            print(f"   {m}: similar folders {near}")
assert WANTED <= folders["photo"] and WANTED <= folders["render"], (
    "Some parts are missing (see above). Fix dataset_folder in classes_50.csv.")
print("All 50 parts found in both photos/ and renders/.")

## Extract the 50 parts

Copies only the selected parts (50) out of the zip. The rest of the parts are not copied.

In [ ]:
from tqdm.auto import tqdm

selected = []
for i in infos:
    p = parse(i.filename)
    if p and p[1] in WANTED:
        selected.append((i, *p))

if SUBSET_DIR.exists():
    shutil.rmtree(SUBSET_DIR)

rows = []
for info, kind, folder, rest in tqdm(selected, desc="extracting"):
    rel = f"{kind}s/{folder}/{rest}" #example: photos/3001/xyz.jpg
    dest = SUBSET_DIR / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    with zf.open(info) as src, open(dest, "wb") as dst:
        shutil.copyfileobj(src, dst)
    rows.append({
        "path": rel,
        "source": kind, #either a real photo or rendered                        
        "class_idx": folder_to_idx[folder],
        "dataset_folder": folder,
        "part_id": folder_to_part[folder],
        "filename": PurePosixPath(rest).name,
    })

images = pd.DataFrame(rows)
print(f"Extracted {len(images):,} images "
      f"({(images.source == 'photo').sum():,} photos, {(images.source == 'render').sum():,} renders)")

## Count images per part

Saved as `results/data_counts.csv`. Flag the parts that have less than 30 photos (MIN_PHOTOS) since there is not enough data to accurately evaluate.

In [ ]:
counts = (images.pivot_table(index="dataset_folder", columns="source", values="path",
                             aggfunc="count", fill_value=0)
          .rename(columns={"photo": "photos", "render": "renders"})
          .reset_index())
counts = classes.merge(counts, on="dataset_folder").sort_values("class_idx")
counts.to_csv(RESULTS_DIR / "data_counts.csv", index=False)

print(counts[["photos", "renders"]].describe().round(0).T)
thin = counts[counts.photos < MIN_PHOTOS]
if len(thin):
    print(f"\n{len(thin)} part(s) have fewer than {MIN_PHOTOS} real photos.")
    print(thin[["class_idx", "dataset_folder", "name", "photos"]].to_string(index=False))
else:
    print(f"\nEvery part has at least {MIN_PHOTOS} real photos.")
counts[["class_idx", "dataset_folder", "name", "photos", "renders"]]

## Split the photos by stage 70/15/15

Real photos are shuffled with a fixed seed and split 70/15/15 within each part (train, dev, eval), so every part has the same proportions in train, dev and test. Renders are all put in train. It doesn't make sense to use renders in dev and evaluation testing.

In [ ]:
import numpy as np

photos = images[images.source == "photo"].copy() #Grab just real photos not renders

rng = np.random.default_rng(SEED) #Use seed so same split even after re-run
photos["split"] = None
for folder, g in photos.groupby("dataset_folder"): #process one part at a time
    idx = rng.permutation(g.index.to_numpy())   # shuffled copy of this part's photo rows
    n_train = round(SPLIT[0] * len(idx))
    n_dev = round(SPLIT[1] * len(idx))
    photos.loc[idx[:n_train], "split"] = "train"
    photos.loc[idx[n_train:n_train + n_dev], "split"] = "dev"
    photos.loc[idx[n_train + n_dev:], "split"] = "test" #gets all the remaining photos (no rounding)

renders = images[images.source == "render"].copy()
renders["split"] = "train" #all renders set to train
all_imgs = pd.concat([photos, renders], ignore_index=True)

cols = ["path", "source", "class_idx", "dataset_folder", "part_id"]
for split in ("train", "dev", "test"):
    all_imgs.loc[all_imgs.split == split, cols].sort_values(cols[:3]).to_csv(
        RESULTS_DIR / "splits" / f"{split}.csv", index=False)

split_counts = pd.crosstab([all_imgs.class_idx, all_imgs.dataset_folder],
                           [all_imgs.split, all_imgs.source])
split_counts.to_csv(RESULTS_DIR / "splits" / "split_counts.csv")

print(pd.crosstab(all_imgs.split, all_imgs.source, margins=True)) #print overall totals
empty = split_counts.loc[:, [("dev", "photo"), ("test", "photo")]].min(axis=1)
if (empty == 0).any():
    print("\nWARNING: some parts have no dev or test photos:", list(empty[empty == 0].index))

## Save the subset to Drive

Packs the extracted images into a single zip on Drive (~0.7 GB). Training copies this one file to the Colab VM
at the start of each session.

In [ ]:
tmp = WORK / "lego50.zip"
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:   # JPEGs are already compressed
    for rel in tqdm(images.path, desc="zipping"):
        z.write(SUBSET_DIR / rel, rel)

out = DRIVE_DIR / "data" / "lego50.zip"
out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(tmp, out)
print(f"Saved {out}  ({out.stat().st_size / 1e9:.2f} GB)")

## Export the result files

The count and split files are saved on Drive at `MyDrive/ece539-p2/results/` and downloaded as `p2_results.zip`.


In [ ]:
from google.colab import files

archive = shutil.make_archive(str(WORK / "p2_results"), "zip", RESULTS_DIR)
print("Results:", sorted(str(p.relative_to(RESULTS_DIR)) for p in RESULTS_DIR.rglob("*") if p.is_file()))
files.download(archive)

## Outputs

* `MyDrive/ece539-p2/data/lego50.zip`: the 50-part images, used by `02_train.ipynb`
* `MyDrive/ece539-p2/results/data_counts.csv`: photos and renders per part
* `MyDrive/ece539-p2/results/splits/{train,dev,test}.csv`: the image list for each split; `split_counts.csv` gives per-part counts

Expected: no part flagged below `MIN_PHOTOS`, and every part has both dev and test photos.